# 03 – Preprocessing: Filtrado de Variables

**Proyecto:** Encuesta Permanente de Empleo Nacional (EPEN) 2024  
**Etapa:** Preprocessing – Paso 3  
**Dataset de entrada:** `epen_outliers_handled.csv` (o fallback a `epen_missing_handled.csv` / `epen_target_defined.csv`)  
**Dataset de salida:** `epen_variable_filtered.csv`

## Objetivo del Notebook

Este notebook define qué variables se conservarán para el modelamiento y cuáles serán excluidas por razones metodológicas.

**Importante:** Este paso **no** selecciona variables por desempeño predictivo, sino por criterios previos:

| Criterio | Descripción |
|---|---|
| Evitar *data leakage* | Eliminar variables que revelan directamente el target |
| Eliminar identificadores | Variables de encuesta, vivienda o diseño muestral |
| Eliminar variables irrelevantes | Constantes, casi constantes o sin utilidad predictiva directa |
| Reducir duplicidades | Columnas redundantes con la misma información |
| Coherencia temática | Mantener variables coherentes con el problema del subempleo por insuficiencia de horas |

### Lo que NO se hace en este notebook:
- No se realiza *train/test split*
- No se realiza balanceo de clases
- No se entrenan modelos
- No se hace *feature selection* estadística
- No se realiza *encoding* ni *scaling* avanzado

---
## 3. Cargar Librerías

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


---
## 4. Cargar Dataset

Se intenta cargar el dataset en el siguiente orden de preferencia:
1. `epen_outliers_handled.csv`
2. `epen_missing_handled.csv`
3. `epen_target_defined.csv`

In [3]:
DATA_PATH = Path('../data/processed')

candidate_files = [
    DATA_PATH / 'epen_outliers_handled.csv',
    DATA_PATH / 'epen_missing_handled.csv',
    DATA_PATH / 'epen_target_defined.csv',
]

df = None
loaded_file = None

for filepath in candidate_files:
    if filepath.exists():
        df = pd.read_csv(filepath)
        loaded_file = filepath.name
        print(f"Dataset cargado: {filepath}")
        break

if df is None:
    raise FileNotFoundError(
        "No se encontró ningún archivo de datos válido en:\n" +
        "\n".join(str(p) for p in candidate_files)
    )

print(f"\nArchivo cargado : {loaded_file}")
print(f"Dimensiones     : {df.shape[0]:,} filas × {df.shape[1]} columnas")

Dataset cargado: ..\data\processed\epen_outliers_handled.csv

Archivo cargado : epen_outliers_handled.csv
Dimensiones     : 24,054 filas × 149 columnas


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14624\2172480777.py:14: DtypeWarning: Columns (0: ESTRATO) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)


In [5]:
# Primeras filas
print("=== Primeras filas ===")
display(df.head())

# Lista de columnas
print(f"\n=== Columnas ({df.shape[1]}) ===")
print(df.columns.tolist())

# Distribución del target
print("\n=== Distribución de target_subempleo_horas ===")
if 'target_subempleo_horas' in df.columns:
    counts = df['target_subempleo_horas'].value_counts()
    pct = df['target_subempleo_horas'].value_counts(normalize=True) * 100
    target_dist = pd.DataFrame({'conteo': counts, 'porcentaje': pct.round(2)})
    display(target_dist)
else:
    print("ADVERTENCIA: 'target_subempleo_horas' no encontrado en el dataset.")

=== Primeras filas ===


,ANIO,MES,CONGLOMERADO,MUESTRA,SELVIV,HOGAR,REGION,LLAVE_PANEL,ESTRATO,C201,C203,C204,C205,C206,C207,C208,C300n,NROINF,C301_DIA,C301_MES,C301_ANIO,C303,C304,C305,C306_1,C306_2,C306_3,C306_4,C306_5,C306_6,C306_7,C306_8,C306_9,C306_10,C306_10A,C306_11,C306A,C308_COD,C309_COD,C310,C311,C312,C313,C317,C317A,C318_1,C318_2,C318_3,C318_4,C318_5,...,C364_3,C365_3,C364_4,C365_4,C366,C366_1,C366_2,C375_1,C375_2,C375_3,C375_4,C375_5,C375_6,C376,C377,OCUP300,I339_1,D341_T,I342,D344,I345_1,I348,D347_T,D350,D351_T,INGTOT,INGTOTP,ingtrabw,RESIDENT,fa_ond24,fa_efm24,fa_amj24,fa_jas24,C208_num,OCUP300_num,P209H_num,target_subempleo_horas,whoraT_missing_flag,C318_T_missing_flag,C208_outlier_flag,whoraT_outlier_flag,INGTOT_outlier_flag,INGTRABW_outlier_flag,INGTOT_log,INGTOTP_log,INGTRABW_log,I339_1_log,I342_log,I345_1_log,I348_log
0,2024,10,18117,2,61,1,1,202410181172006016,1,1,1,1,2,,2,60.0000,1,1,19.0000,5.0000,1964,1,,,,,,,,,,,,,,,,9411,5610,2,,3,2,1,1,0,6,6,6,6,...,2,,2,,11,5,,2,2,2,2,2,2,10,7,1,NaN,,1367.0000,,NaN,NaN,,,,1367.0000,1367.0000,1367.0000,1,700.4007,NaN,NaN,NaN,60.0000,1.0000,2.0000,0,0,0,0,0,0,0,7.2211,7.2211,7.2211,NaN,7.2211,NaN,NaN
1,2024,10,18117,2,61,1,1,202410181172006016,1,2,7,1,2,,2,49.0000,2,2,30.0000,11.0000,1974,1,,,,,,,,,,,,,,,,9411,5610,2,,3,2,1,1,0,5,5,5,5,...,2,,2,,6,5,,2,2,2,2,2,2,10,7,1,NaN,,1404.0000,130,NaN,NaN,,,,1534.0000,1534.0000,1534.0000,1,700.4007,NaN,NaN,NaN,49.0000,1.0000,1.0000,1,0,0,0,0,0,0,7.3363,7.3363,7.3363,NaN,7.2478,NaN,NaN
2,2024,10,1823302,1,59,1,1,20241018233021001728,1,1,1,1,2,,1,52.0000,1,2,14.0000,9.0000,1972,1,,,,,,,,,,,,,,,,8322,4922,2,,3,2,1,1,0,8,8,8,8,...,2,,2,,6,5,,2,2,2,2,2,2,10,7,1,NaN,,1400.0000,,NaN,NaN,,,,1400.0000,1400.0000,1400.0000,1,896.4695,NaN,NaN,NaN,52.0000,1.0000,2.0000,0,0,0,0,0,0,0,7.2449,7.2449,7.2449,NaN,7.2449,NaN,NaN
3,2024,10,1823302,1,59,1,1,20241018233021001728,1,2,2,1,2,,2,51.0000,2,2,10.0000,10.0000,1972,1,,,,,,,,,,,,,,,,5120,5610,3,5,2,2,1,3,12,12,12,12,0,...,2,,2,,6,5,,2,2,2,2,2,2,1,1,1,1516.0000,,NaN,,NaN,NaN,,,,1516.0000,1516.0000,1516.0000,1,844.0361,NaN,NaN,NaN,51.0000,1.0000,2.0000,0,0,0,0,0,0,0,7.3245,7.3245,7.3245,7.3245,NaN,NaN,NaN
4,2024,10,1823302,1,60,1,1,20241018233021001728,1,1,1,1,2,,2,43.0000,1,1,25.0000,4.0000,1981,1,,,,,,,,,,,,,,,,4419,8610,3,5,2,2,1,3,0,5,5,5,5,...,2,,2,,6,5,,2,2,2,2,2,2,10,1,1,1083.0000,,NaN,,NaN,NaN,,,,1083.0000,1083.0000,1083.0000,1,1017.6463,NaN,NaN,NaN,43.0000,1.0000,2.0000,0,0,0,0,0,0,0,6.9884,6.9884,6.9884,6.9884,NaN,NaN,NaN



=== Columnas (149) ===
['ANIO', 'MES', 'CONGLOMERADO', 'MUESTRA', 'SELVIV', 'HOGAR', 'REGION', 'LLAVE_PANEL', 'ESTRATO', 'C201', 'C203', 'C204', 'C205', 'C206', 'C207', 'C208', 'C300n', 'NROINF', 'C301_DIA', 'C301_MES', 'C301_ANIO', 'C303', 'C304', 'C305', 'C306_1', 'C306_2', 'C306_3', 'C306_4', 'C306_5', 'C306_6', 'C306_7', 'C306_8', 'C306_9', 'C306_10', 'C306_10A', 'C306_11', 'C306A', 'C308_COD', 'C309_COD', 'C310', 'C311', 'C312', 'C313', 'C317', 'C317A', 'C318_1', 'C318_2', 'C318_3', 'C318_4', 'C318_5', 'C318_6', 'C318_7', 'C318_T', 'C328_T', 'whoraT', 'C330', 'C331', 'C333', 'C334', 'P209H', 'C335', 'C338', 'C339_1', 'C341_T', 'C342', 'C344', 'C345_1', 'C347_T', 'C348', 'C350', 'C352', 'C353', 'C354', 'C355', 'C356', 'C357_I', 'C358', 'C359', 'SEGURO1', 'C361_1', 'C362_1', 'C361_2', 'C362_2', 'C361_3', 'C362_3', 'C361_4', 'C362_4', 'C361_5', 'C362_5', 'C361_6', 'C362_6', 'C361_7', 'C362_7', 'C361_8', 'C362_8', 'C364_1', 'C365_1', 'C364_2', 'C365_2', 'C364_3', 'C365_3', 'C364_4', 

,conteo,porcentaje
target_subempleo_horas,,
0,18064,75.1000
1,5990,24.9000


---
## 5. Validación Inicial

Se confirma que el dataset corresponde al universo analítico esperado:
- Residentes habituales, personas de **14 años a más** y **ocupadas**.
- El target `target_subempleo_horas` existe y no tiene valores nulos.

In [6]:
# Confirmar que existe target_subempleo_horas
assert 'target_subempleo_horas' in df.columns, \
    "ERROR: 'target_subempleo_horas' no existe en el dataset."
print("target_subempleo_horas encontrado: OK")

# Confirmar que el target no tiene valores nulos
nulos_target = df['target_subempleo_horas'].isnull().sum()
assert nulos_target == 0, \
    f"ERROR: target_subempleo_horas tiene {nulos_target} valores nulos."
print(f"target_subempleo_horas sin valores nulos: OK ({df.shape[0]:,} registros)")

# Resumen general
print(f"\nTipos de dato en el dataset:")
print(df.dtypes.value_counts())
print(f"\nDimensiones del dataset de entrada: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print("Validación inicial completada.")

target_subempleo_horas encontrado: OK
target_subempleo_horas sin valores nulos: OK (24,054 registros)

Tipos de dato en el dataset:
int64      62
str        57
float64    28
object      2
Name: count, dtype: int64

Dimensiones del dataset de entrada: 24,054 filas × 149 columnas
Validación inicial completada.


---
## 6. Crear Copia de Trabajo

Se crea una copia de trabajo `df_filtered` para aplicar los filtros sin modificar el dataset original.

In [7]:
df_filtered = df.copy()
print(f"Copia de trabajo creada: {df_filtered.shape[0]:,} filas × {df_filtered.shape[1]} columnas")

Copia de trabajo creada: 24,054 filas × 149 columnas


---
## 7. Identificar Variables de Data Leakage

Las siguientes variables **no deben entrar como predictores** porque fueron usadas para construir el target o revelan directamente la condición de subempleo:

| Variable | Razón de exclusión |
|---|---|
| `P209H` | Variable original utilizada para construir `target_subempleo_horas` |
| `C333` | Pregunta directamente si la persona quería trabajar más horas |
| `C334` | Pregunta directamente si la persona estuvo disponible para trabajar más horas |

Usar estas variables sería *data leakage*: el modelo "sabría" el resultado antes de la predicción.

In [8]:
leakage_variables = ['P209H', 'C333', 'C334']

leakage_present = [col for col in leakage_variables if col in df_filtered.columns]
leakage_absent  = [col for col in leakage_variables if col not in df_filtered.columns]

print("=== Variables de data leakage ===")
print(f"Variables definidas : {leakage_variables}")
print(f"Presentes en el df  : {leakage_present}")
print(f"No encontradas      : {leakage_absent}")

if leakage_present:
    df_filtered.drop(columns=leakage_present, inplace=True)
    print(f"\nEliminadas: {leakage_present}")
else:
    print("\nNinguna variable de leakage presente en el dataset.")

print(f"\nDimensiones después de eliminar leakage: {df_filtered.shape[0]:,} × {df_filtered.shape[1]}")

=== Variables de data leakage ===
Variables definidas : ['P209H', 'C333', 'C334']
Presentes en el df  : ['P209H', 'C333', 'C334']
No encontradas      : []

Eliminadas: ['P209H', 'C333', 'C334']

Dimensiones después de eliminar leakage: 24,054 × 146


---
## 8. Identificar Variables Administrativas o de Identificación

Las siguientes variables **identifican encuesta, vivienda, hogar, persona o diseño muestral**.  
No representan características laborales o sociodemográficas interpretables para la predicción individual.

| Variable | Descripción |
|---|---|
| `ANIO` | Año del relevamiento |
| `MES` | Mes del relevamiento |
| `CONGLOMERADO` | Identificador de conglomerado muestral |
| `MUESTRA` | Identificador de muestra |
| `SELVIV` | Número de vivienda seleccionada |
| `HOGAR` | Número de hogar |
| `LLAVE_PANEL` | Identificador del panel longitudinal |
| `C201` | Número de orden de la persona en el hogar |
| `C300n` | Número del formulario de actividad |
| `NROINF` | Número del informante |

In [9]:
id_variables = [
    'ANIO', 'MES', 'CONGLOMERADO', 'MUESTRA', 'SELVIV',
    'HOGAR', 'LLAVE_PANEL', 'C201', 'C300n', 'NROINF'
]

id_present = [col for col in id_variables if col in df_filtered.columns]
id_absent   = [col for col in id_variables if col not in df_filtered.columns]

print("=== Variables administrativas / identificadores ===")
print(f"Definidas  : {id_variables}")
print(f"Presentes  : {id_present}")
print(f"No halladas: {id_absent}")

if id_present:
    df_filtered.drop(columns=id_present, inplace=True)
    print(f"\nEliminadas: {id_present}")
else:
    print("\nNinguna variable administrativa presente en el dataset.")

print(f"\nDimensiones después de eliminar identificadores: {df_filtered.shape[0]:,} × {df_filtered.shape[1]}")

=== Variables administrativas / identificadores ===
Definidas  : ['ANIO', 'MES', 'CONGLOMERADO', 'MUESTRA', 'SELVIV', 'HOGAR', 'LLAVE_PANEL', 'C201', 'C300n', 'NROINF']
Presentes  : ['ANIO', 'MES', 'CONGLOMERADO', 'MUESTRA', 'SELVIV', 'HOGAR', 'LLAVE_PANEL', 'C201', 'C300n', 'NROINF']
No halladas: []

Eliminadas: ['ANIO', 'MES', 'CONGLOMERADO', 'MUESTRA', 'SELVIV', 'HOGAR', 'LLAVE_PANEL', 'C201', 'C300n', 'NROINF']

Dimensiones después de eliminar identificadores: 24,054 × 136


---
## 9. Tratamiento del Factor de Expansión (`fa_son24`)

`fa_son24` es el factor de expansión muestral de la EPEN 2024.

- **No se usará como predictor** del modelo base, porque no representa una característica individual de la persona o su situación laboral.
- Puede conservarse en un archivo auxiliar para análisis descriptivo ponderado (estimaciones a nivel poblacional).

> En esta etapa se extrae como variable auxiliar y luego se elimina del dataset de modelamiento.

In [10]:
expansion_col = 'fa_son24'

if expansion_col in df_filtered.columns:
    expansion_factor = df_filtered[expansion_col].copy()
    print(f"'{expansion_col}' encontrado. Guardando como variable auxiliar.")
    print(f"  min   : {expansion_factor.min():,.4f}")
    print(f"  max   : {expansion_factor.max():,.4f}")
    print(f"  media : {expansion_factor.mean():,.4f}")
    df_filtered.drop(columns=[expansion_col], inplace=True)
    print(f"\n'{expansion_col}' eliminado del dataset de modelamiento.")
else:
    expansion_factor = None
    print(f"'{expansion_col}' no encontrado en el dataset.")

print(f"\nDimensiones: {df_filtered.shape[0]:,} × {df_filtered.shape[1]}")

'fa_son24' no encontrado en el dataset.

Dimensiones: 24,054 × 136


---
## 10. Identificar Variables Duplicadas o Redundantes

Se revisan variables de ingresos y horas que podrían contener información similar entre sí.

| Variable | Descripción tentativa |
|---|---|
| `INGTOT` | Ingreso total del trabajo principal |
| `INGTOTP` | Ingreso total del trabajo principal (otra codificación) |
| `INGTRABW` | Ingreso laboral semanal |
| `ingtrabw` | Mismo campo con nombre en minúsculas |
| `I339_1` | Componente de ingresos del trabajo dependiente |
| `I342` | Componente de ingresos del trabajo independiente |
| `I345_1` | Componente de ingresos secundario |
| `I348` | Componente de otros ingresos |
| `C318_T` | Horas habituales en trabajo principal |
| `C328_T` | Horas habituales en otros trabajos |
| `whoraT` | Total de horas habituales (suma de trabajos) |
| `C331` | Horas que desearía trabajar |

Si `INGTRABW` e `ingtrabw` contienen exactamente los mismos valores, se conserva solo una.  
Los pares altamente correlacionados (> 0.90) se documentan para revisión en *feature selection*.

In [11]:
redundant_candidates = [
    'INGTOT', 'INGTOTP', 'INGTRABW', 'ingtrabw',
    'I339_1', 'I342', 'I345_1', 'I348',
    'C318_T', 'C328_T', 'whoraT', 'C331'
]

present_candidates = [c for c in redundant_candidates if c in df_filtered.columns]
print(f"Variables candidatas presentes ({len(present_candidates)}): {present_candidates}")

# ── 1. Verificar si INGTRABW e ingtrabw son idénticas ──────────────────────
duplicated_exact = []
if 'INGTRABW' in df_filtered.columns and 'ingtrabw' in df_filtered.columns:
    are_equal = df_filtered['INGTRABW'].equals(df_filtered['ingtrabw'])
    if are_equal:
        print("\nINGTRABW e ingtrabw son IDÉNTICAS → se conserva 'ingtrabw', se elimina 'INGTRABW'")
        df_filtered.drop(columns=['INGTRABW'], inplace=True)
        duplicated_exact.append('INGTRABW')
    else:
        print("\nINGTRABW e ingtrabw tienen valores DISTINTOS → se conservan ambas para revisión.")

# ── 2. Tabla de correlación entre variables numéricas candidatas ────────────
numeric_candidates = [
    c for c in present_candidates
    if c in df_filtered.columns and pd.api.types.is_numeric_dtype(df_filtered[c])
]

if len(numeric_candidates) >= 2:
    print(f"\nCorrelación entre variables numéricas candidatas ({len(numeric_candidates)}):")
    corr_matrix = df_filtered[numeric_candidates].corr().round(3)
    display(corr_matrix)

    # Pares con correlación > 0.90
    high_corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i + 1, len(corr_matrix.columns)):
            val = abs(corr_matrix.iloc[i, j])
            if val > 0.90:
                high_corr_pairs.append({
                    'var_1': corr_matrix.columns[i],
                    'var_2': corr_matrix.columns[j],
                    'correlacion': corr_matrix.iloc[i, j]
                })
    if high_corr_pairs:
        print("\nPares con correlación > 0.90 (a revisar en feature selection):")
        display(pd.DataFrame(high_corr_pairs))
    else:
        print("\nNo se encontraron pares con correlación > 0.90.")
else:
    print("Insuficientes variables numéricas para calcular correlación.")

print(f"\nDimensiones: {df_filtered.shape[0]:,} × {df_filtered.shape[1]}")

Variables candidatas presentes (11): ['INGTOT', 'INGTOTP', 'ingtrabw', 'I339_1', 'I342', 'I345_1', 'I348', 'C318_T', 'C328_T', 'whoraT', 'C331']

Correlación entre variables numéricas candidatas (11):


,INGTOT,INGTOTP,ingtrabw,I339_1,I342,I345_1,I348,C318_T,C328_T,whoraT,C331
INGTOT,1.0000,0.9740,0.9860,0.9600,0.9860,0.8750,0.8170,0.1420,0.0470,0.1810,0.2010
INGTOTP,0.9740,1.0000,0.9660,0.9940,0.9960,0.6320,0.4920,0.1590,-0.0310,0.1680,0.1940
ingtrabw,0.9860,0.9660,1.0000,0.9560,0.9860,0.8590,0.8030,0.1310,0.0390,0.1690,0.2010
I339_1,0.9600,0.9940,0.9560,1.0000,NaN,0.6330,0.4820,0.0740,-0.0100,0.0920,0.1070
I342,0.9860,0.9960,0.9860,NaN,1.0000,0.4170,0.5320,0.2170,-0.0430,0.2110,0.2140
I345_1,0.8750,0.6320,0.8590,0.6330,0.4170,1.0000,0.9640,0.0320,0.1920,0.1580,0.0380
I348,0.8170,0.4920,0.8030,0.4820,0.5320,0.9640,1.0000,0.0360,0.1750,0.1460,0.1580
C318_T,0.1420,0.1590,0.1310,0.0740,0.2170,0.0320,0.0360,1.0000,-0.1010,0.8580,0.5360
C328_T,0.0470,-0.0310,0.0390,-0.0100,-0.0430,0.1920,0.1750,-0.1010,1.0000,0.4940,0.3090
whoraT,0.1810,0.1680,0.1690,0.0920,0.2110,0.1580,0.1460,0.8580,0.4940,1.0000,0.9500



Pares con correlación > 0.90 (a revisar en feature selection):


,var_1,var_2,correlacion
0,INGTOT,INGTOTP,0.9740
1,INGTOT,ingtrabw,0.9860
2,INGTOT,I339_1,0.9600
3,INGTOT,I342,0.9860
4,INGTOTP,ingtrabw,0.9660
5,INGTOTP,I339_1,0.9940
6,INGTOTP,I342,0.9960
7,ingtrabw,I339_1,0.9560
8,ingtrabw,I342,0.9860
9,I345_1,I348,0.9640



Dimensiones: 24,054 × 136


---
## 11. Identificar Columnas con Alto Porcentaje de Valores Faltantes

Se calcula el porcentaje de nulos por columna y se aplica un umbral del **60%**.

- Si una columna supera el 60% de nulos y **no es conceptualmente importante**, se elimina.
- Si la variable es importante (ingresos, horas, empleo), se mantiene para evaluación en *feature engineering* o *feature selection*.

In [12]:
missing_threshold = 0.60

# Tabla de nulos por columna
missing_df = pd.DataFrame({
    'columna'         : df_filtered.columns,
    'cantidad_nulos'  : df_filtered.isnull().sum().values,
    'porcentaje_nulos': (df_filtered.isnull().sum() / len(df_filtered)).round(4).values,
    'tipo_dato'       : df_filtered.dtypes.values,
}).sort_values('porcentaje_nulos', ascending=False).reset_index(drop=True)

print("=== Columnas con valores faltantes ===")
display(missing_df[missing_df['cantidad_nulos'] > 0])

# Variables con alto missing (excluyendo target)
high_missing_variables = missing_df[
    (missing_df['porcentaje_nulos'] > missing_threshold) &
    (missing_df['columna'] != 'target_subempleo_horas')
]['columna'].tolist()

print(f"\nVariables con más del {missing_threshold*100:.0f}% de nulos ({len(high_missing_variables)}):")
print(high_missing_variables)

# Variables conceptualmente importantes (no eliminar automáticamente)
important_vars = [
    'INGTOT', 'INGTOTP', 'INGTRABW', 'ingtrabw',
    'I339_1', 'I342', 'I345_1', 'I348',
    'C318_T', 'C328_T', 'whoraT', 'C331',
    'C366', 'C366_1', 'C366_2', 'SEGURO1',
]

high_missing_safe_to_drop = [v for v in high_missing_variables if v not in important_vars]
high_missing_keep         = [v for v in high_missing_variables if v in important_vars]

print(f"\nVariables con alto missing que se pueden eliminar ({len(high_missing_safe_to_drop)}): {high_missing_safe_to_drop}")
print(f"Variables con alto missing pero importantes, se mantienen ({len(high_missing_keep)}): {high_missing_keep}")

if high_missing_safe_to_drop:
    df_filtered.drop(columns=high_missing_safe_to_drop, inplace=True)
    print(f"\nEliminadas {len(high_missing_safe_to_drop)} columnas con alto missing no importantes.")
else:
    print("\nNo se eliminaron columnas por este criterio.")

print(f"\nDimensiones: {df_filtered.shape[0]:,} × {df_filtered.shape[1]}")

=== Columnas con valores faltantes ===


,columna,cantidad_nulos,porcentaje_nulos,tipo_dato
0,I345_1_log,23491,0.9766,float64
1,I345_1,23491,0.9766,float64
2,I348,22516,0.9361,float64
3,I348_log,22516,0.9361,float64
4,C328_T,21604,0.8981,float64
5,C331,20614,0.8570,float64
6,fa_ond24,18334,0.7622,float64
7,fa_jas24,18062,0.7509,float64
8,fa_amj24,17885,0.7435,float64
9,fa_efm24,17881,0.7434,float64



Variables con más del 60% de nulos (12):
['I345_1_log', 'I345_1', 'I348', 'I348_log', 'C328_T', 'C331', 'fa_ond24', 'fa_jas24', 'fa_amj24', 'fa_efm24', 'I342', 'I342_log']

Variables con alto missing que se pueden eliminar (7): ['I345_1_log', 'I348_log', 'fa_ond24', 'fa_jas24', 'fa_amj24', 'fa_efm24', 'I342_log']
Variables con alto missing pero importantes, se mantienen (5): ['I345_1', 'I348', 'C328_T', 'C331', 'I342']

Eliminadas 7 columnas con alto missing no importantes.

Dimensiones: 24,054 × 129


---
## 12. Identificar Columnas Constantes o Casi Constantes

- **Columnas constantes** (1 valor único): se eliminan automáticamente, no aportan información.  
- **Columnas casi constantes** (valor más frecuente ≥ 99% de los casos): se documentan para revisión. No se eliminan automáticamente.

In [13]:
# Columnas constantes (1 valor único, excluyendo target)
constant_variables = [
    col for col in df_filtered.columns
    if df_filtered[col].nunique(dropna=True) <= 1
    and col != 'target_subempleo_horas'
]

# Columnas casi constantes (valor más frecuente >= 99%)
quasi_constant_variables = []
for col in df_filtered.columns:
    if col in constant_variables or col == 'target_subempleo_horas':
        continue
    if df_filtered[col].nunique(dropna=True) > 1:
        top_freq = df_filtered[col].value_counts(normalize=True, dropna=True).iloc[0]
        if top_freq >= 0.99:
            quasi_constant_variables.append({
                'columna'              : col,
                'valor_mas_frecuente'  : df_filtered[col].value_counts().index[0],
                'frecuencia_relativa'  : round(top_freq, 4),
            })

print(f"=== Columnas constantes ({len(constant_variables)}) ===")
print(constant_variables)

print(f"\n=== Columnas casi constantes ({len(quasi_constant_variables)}) ===")
if quasi_constant_variables:
    display(pd.DataFrame(quasi_constant_variables))
else:
    print("No se encontraron columnas casi constantes.")

# Eliminar constantes
if constant_variables:
    df_filtered.drop(columns=constant_variables, inplace=True)
    print(f"\nEliminadas {len(constant_variables)} columnas constantes.")
else:
    print("\nNo hay columnas constantes para eliminar.")

print(f"\nDimensiones: {df_filtered.shape[0]:,} × {df_filtered.shape[1]}")

=== Columnas constantes (8) ===
['REGION', 'C306A', 'C361_7', 'C362_7', 'OCUP300', 'RESIDENT', 'OCUP300_num', 'C208_outlier_flag']

=== Columnas casi constantes (28) ===


,columna,valor_mas_frecuente,frecuencia_relativa
0,C204,1,0.9960
1,C205,2,0.9960
2,C206,,0.9960
3,C344,,0.9988
4,C347_T,,0.9941
5,C352,,0.9978
6,C353,,0.9978
7,C354,,0.9978
8,C355,,0.9978
9,C356,,0.9978



Eliminadas 8 columnas constantes.

Dimensiones: 24,054 × 121


---
## 13. Identificar Variables Categóricas de Alta Cardinalidad

Las variables con muchos valores únicos (ej.: códigos de ocupación o actividad económica) pueden generar demasiadas columnas en *one-hot encoding*, saturando el espacio de características.

| Variable | Descripción tentativa |
|---|---|
| `C308_COD` | Código de ocupación principal (CIUO) |
| `C309_COD` | Código de actividad económica (ISIC/CIIU) |

### Decisión para el modelo base:
- `C308_COD` y `C309_COD` se **excluyen** del modelo base si existen.
- Podrían recuperarse en una versión avanzada mediante agrupación por **familia ocupacional** (gran grupo CIUO) o **sector económico** (sección ISIC).

In [14]:
high_cardinality_vars = ['C308_COD', 'C309_COD']

hc_present = [col for col in high_cardinality_vars if col in df_filtered.columns]
hc_absent   = [col for col in high_cardinality_vars if col not in df_filtered.columns]

print("=== Variables de alta cardinalidad ===")
for col in hc_present:
    n_unique = df_filtered[col].nunique()
    print(f"  {col}: {n_unique} valores únicos")
print(f"No encontradas: {hc_absent}")

if hc_present:
    df_filtered.drop(columns=hc_present, inplace=True)
    print(f"\nEliminadas del modelo base: {hc_present}")
    print("Nota: pueden recuperarse en versión avanzada con agrupación por sector/familia ocupacional.")
else:
    print("\nNo hay variables de alta cardinalidad para eliminar.")

print(f"\nDimensiones: {df_filtered.shape[0]:,} × {df_filtered.shape[1]}")

=== Variables de alta cardinalidad ===
  C308_COD: 400 valores únicos
  C309_COD: 343 valores únicos
No encontradas: []

Eliminadas del modelo base: ['C308_COD', 'C309_COD']
Nota: pueden recuperarse en versión avanzada con agrupación por sector/familia ocupacional.

Dimensiones: 24,054 × 119


---
## 14. Definir Variables Candidatas Finales por Grupo

Se definen listas de variables candidatas organizadas por categoría temática.  
Solo se conservarán las que realmente existen en el DataFrame en este punto.

In [16]:
# ── Grupos temáticos de variables candidatas ────────────────────────────────
demographic_variables = ['C207', 'C208', 'C203']

education_variables = ['C366', 'C366_1', 'C366_2']

employment_variables = ['C310', 'C311', 'C312', 'C313', 'C317', 'C317A', 'C335']

hours_variables = ['C318_T', 'C328_T', 'whoraT', 'C331']

income_variables = [
    'INGTOT', 'INGTOTP', 'INGTRABW', 'ingtrabw',
    'I339_1', 'I342', 'I345_1', 'I348',
    'INGTOT_log', 'INGTOTP_log', 'INGTRABW_log', 'ingtrabw_log',
]

social_protection_variables = ['SEGURO1', 'C361_1', 'C361_5', 'C364_1', 'C364_2']

disability_variables = ['C375_1', 'C375_2', 'C375_3', 'C375_4', 'C375_5', 'C375_6']

ethnicity_variables = ['C376', 'C377']

# Variables escaladas (generadas en etapas anteriores)
scaled_variables = [
    col for col in df_filtered.columns
    if col.endswith('_std') or col.endswith('_minmax') or col.endswith('_robust')
]

# Flags de missing
missing_flags = [col for col in df_filtered.columns if col.endswith('_missing_flag')]

# Flags de outliers
outlier_flags = [col for col in df_filtered.columns if col.endswith('_outlier_flag')]

# ── Construir lista de candidatas verificando existencia ────────────────────
all_candidate_groups = {
    'demograficas'         : demographic_variables,
    'educacion'            : education_variables,
    'empleo'               : employment_variables,
    'horas'                : hours_variables,
    'ingresos'             : income_variables,
    'proteccion_social'    : social_protection_variables,
    'discapacidad'         : disability_variables,
    'etnicidad'            : ethnicity_variables,
    'escaladas'            : scaled_variables,
    'flags_missing'        : missing_flags,
    'flags_outliers'       : outlier_flags,
}

candidate_variables = []
for group_name, group_vars in all_candidate_groups.items():
    present = [v for v in group_vars if v in df_filtered.columns]
    absent  = [v for v in group_vars if v not in df_filtered.columns]
    print(f"\n{group_name.upper()} ({len(present)} de {len(group_vars)} encontradas)")
    print(f"  Presentes : {present}")
    if absent:
        print(f"  Ausentes  : {absent}")
    candidate_variables.extend(present)

# Eliminar duplicados manteniendo orden
candidate_variables = list(dict.fromkeys(candidate_variables))

print(f"\n{'='*50}")
print(f"Total de variables candidatas: {len(candidate_variables)}")
print(candidate_variables)


DEMOGRAFICAS (3 de 3 encontradas)
  Presentes : ['C207', 'C208', 'C203']

EDUCACION (3 de 3 encontradas)
  Presentes : ['C366', 'C366_1', 'C366_2']

EMPLEO (7 de 7 encontradas)
  Presentes : ['C310', 'C311', 'C312', 'C313', 'C317', 'C317A', 'C335']

HORAS (4 de 4 encontradas)
  Presentes : ['C318_T', 'C328_T', 'whoraT', 'C331']

INGRESOS (10 de 12 encontradas)
  Presentes : ['INGTOT', 'INGTOTP', 'ingtrabw', 'I339_1', 'I342', 'I345_1', 'I348', 'INGTOT_log', 'INGTOTP_log', 'INGTRABW_log']
  Ausentes  : ['INGTRABW', 'ingtrabw_log']

PROTECCION_SOCIAL (5 de 5 encontradas)
  Presentes : ['SEGURO1', 'C361_1', 'C361_5', 'C364_1', 'C364_2']

DISCAPACIDAD (6 de 6 encontradas)
  Presentes : ['C375_1', 'C375_2', 'C375_3', 'C375_4', 'C375_5', 'C375_6']

ETNICIDAD (2 de 2 encontradas)
  Presentes : ['C376', 'C377']

ESCALADAS (0 de 0 encontradas)
  Presentes : []

FLAGS_MISSING (2 de 2 encontradas)
  Presentes : ['whoraT_missing_flag', 'C318_T_missing_flag']

FLAGS_OUTLIERS (3 de 3 encontradas)
  

---
## 15. Construir Dataset Filtrado

Se construye `df_model_base` conservando únicamente las variables candidatas más el target.

In [17]:
# Columnas a conservar: candidatas + target
columns_to_keep = candidate_variables + ['target_subempleo_horas']

# Eliminar duplicados manteniendo orden
columns_to_keep = list(dict.fromkeys(columns_to_keep))

# Verificar que todas existan
missing_cols = [c for c in columns_to_keep if c not in df_filtered.columns]
if missing_cols:
    print(f"ADVERTENCIA: Las siguientes columnas no existen y serán ignoradas: {missing_cols}")
    columns_to_keep = [c for c in columns_to_keep if c in df_filtered.columns]

# Construir dataset filtrado
df_model_base = df_filtered[columns_to_keep].copy()

print(f"Dataset base para modelamiento:")
print(f"  Filas    : {df_model_base.shape[0]:,}")
print(f"  Columnas : {df_model_base.shape[1]}")
print(f"\nPrimeras filas:")
display(df_model_base.head())

Dataset base para modelamiento:
  Filas    : 24,054
  Columnas : 46

Primeras filas:


,C207,C208,C203,C366,C366_1,C366_2,C310,C311,C312,C313,C317,C317A,C335,C318_T,C328_T,whoraT,C331,INGTOT,INGTOTP,ingtrabw,I339_1,I342,I345_1,I348,INGTOT_log,INGTOTP_log,INGTRABW_log,SEGURO1,C361_1,C361_5,C364_1,C364_2,C375_1,C375_2,C375_3,C375_4,C375_5,C375_6,C376,C377,whoraT_missing_flag,C318_T_missing_flag,whoraT_outlier_flag,INGTOT_outlier_flag,INGTRABW_outlier_flag,target_subempleo_horas
0,2,60.0000,1,11,5,,2,,3,2,1,1,2,36.0000,NaN,36.0000,NaN,1367.0000,1367.0000,1367.0000,NaN,1367.0000,NaN,NaN,7.2211,7.2211,7.2211,5,2,1,2,2,2,2,2,2,2,2,10,7,0,0,0,0,0,0
1,2,49.0000,7,6,5,,2,,3,2,1,1,2,30.0000,NaN,30.0000,NaN,1534.0000,1534.0000,1534.0000,NaN,1404.0000,NaN,NaN,7.3363,7.3363,7.3363,5,2,1,2,2,2,2,2,2,2,2,10,7,0,0,0,0,0,1
2,1,52.0000,1,6,5,,2,,3,2,1,1,2,40.0000,NaN,40.0000,NaN,1400.0000,1400.0000,1400.0000,NaN,1400.0000,NaN,NaN,7.2449,7.2449,7.2449,5,2,1,1,2,2,2,2,2,2,2,10,7,0,0,0,0,0,0
3,2,51.0000,2,6,5,,3,5,2,2,1,3,2,72.0000,NaN,72.0000,NaN,1516.0000,1516.0000,1516.0000,1516.0000,NaN,NaN,NaN,7.3245,7.3245,7.3245,5,2,1,2,2,2,2,2,2,2,2,1,1,0,0,0,0,0,0
4,2,43.0000,1,6,5,,3,5,2,2,1,3,2,30.0000,NaN,30.0000,NaN,1083.0000,1083.0000,1083.0000,1083.0000,NaN,NaN,NaN,6.9884,6.9884,6.9884,6,2,2,2,2,2,2,2,2,2,2,10,1,0,0,0,0,0,0


---
## 16. Validaciones Finales

Se confirma que el dataset filtrado cumple con todos los requisitos:
- El target existe y no está contaminado con variables de leakage.
- No están presentes identificadores administrativos ni el factor de expansión.
- No hay columnas duplicadas.
- El dataset tiene filas y columnas suficientes.

In [18]:
print("=== VALIDACIONES FINALES ===\n")

# 1. Target existe
assert 'target_subempleo_horas' in df_model_base.columns, \
    "ERROR: 'target_subempleo_horas' no encontrado en df_model_base"
print("1. target_subempleo_horas presente: OK")

# 2. Variables de leakage NO están
for lv in ['P209H', 'C333', 'C334']:
    assert lv not in df_model_base.columns, f"ERROR: '{lv}' (leakage) está en df_model_base"
print("2. Variables de data leakage ausentes: OK")

# 3. Identificadores administrativos NO están
admin_in_model = [
    col for col in ['ANIO', 'MES', 'CONGLOMERADO', 'MUESTRA', 'SELVIV',
                    'HOGAR', 'LLAVE_PANEL', 'C201', 'C300n', 'NROINF']
    if col in df_model_base.columns
]
if admin_in_model:
    print(f"   ADVERTENCIA: Identificadores presentes: {admin_in_model}")
else:
    print("3. Identificadores administrativos ausentes: OK")

# 4. Factor de expansión NO está
assert 'fa_son24' not in df_model_base.columns, "ERROR: 'fa_son24' está en df_model_base"
print("4. Factor de expansión (fa_son24) ausente: OK")

# 5. Sin columnas duplicadas
n_dup = df_model_base.columns.duplicated().sum()
assert n_dup == 0, f"ERROR: {n_dup} columnas duplicadas"
print("5. Sin columnas duplicadas: OK")

# 6. Dataset con tamaño suficiente
assert df_model_base.shape[0] > 0, "ERROR: el dataset no tiene filas."
assert df_model_base.shape[1] >= 3, "ERROR: el dataset tiene menos de 3 columnas."
print(f"6. Dataset con {df_model_base.shape[0]:,} filas y {df_model_base.shape[1]} columnas: OK")

# ── Resumen comparativo ──────────────────────────────────────────────────────
n_original   = df.shape[1]
n_filtrado   = df_model_base.shape[1]
n_eliminadas = n_original - n_filtrado

print(f"\n{'='*50}")
print(f"{'Dimensiones originales':<30}: {df.shape[0]:>6,} × {n_original}")
print(f"{'Dimensiones filtradas':<30}: {df_model_base.shape[0]:>6,} × {n_filtrado}")
print(f"{'Variables eliminadas':<30}: {n_eliminadas:>6}")
print(f"{'='*50}")

print("\n=== Variables conservadas ===")
print(sorted(df_model_base.columns.tolist()))

# Variables eliminadas
all_removed = (
    leakage_present
    + id_present
    + ([expansion_col] if expansion_factor is not None else [])
    + (duplicated_exact if 'duplicated_exact' in dir() else [])
    + (high_missing_safe_to_drop if 'high_missing_safe_to_drop' in dir() else [])
    + constant_variables
    + hc_present
)
print(f"\n=== Variables eliminadas ({len(all_removed)}) ===")
print(all_removed)

=== VALIDACIONES FINALES ===

1. target_subempleo_horas presente: OK
2. Variables de data leakage ausentes: OK
3. Identificadores administrativos ausentes: OK
4. Factor de expansión (fa_son24) ausente: OK
5. Sin columnas duplicadas: OK
6. Dataset con 24,054 filas y 46 columnas: OK

Dimensiones originales        : 24,054 × 149
Dimensiones filtradas         : 24,054 × 46
Variables eliminadas          :    103

=== Variables conservadas ===
['C203', 'C207', 'C208', 'C310', 'C311', 'C312', 'C313', 'C317', 'C317A', 'C318_T', 'C318_T_missing_flag', 'C328_T', 'C331', 'C335', 'C361_1', 'C361_5', 'C364_1', 'C364_2', 'C366', 'C366_1', 'C366_2', 'C375_1', 'C375_2', 'C375_3', 'C375_4', 'C375_5', 'C375_6', 'C376', 'C377', 'I339_1', 'I342', 'I345_1', 'I348', 'INGTOT', 'INGTOTP', 'INGTOTP_log', 'INGTOT_log', 'INGTOT_outlier_flag', 'INGTRABW_log', 'INGTRABW_outlier_flag', 'SEGURO1', 'ingtrabw', 'target_subempleo_horas', 'whoraT', 'whoraT_missing_flag', 'whoraT_outlier_flag']

=== Variables eliminadas 

---
## 17. Guardar Resultados

Se guardan los siguientes archivos en `../data/processed/`:

| Archivo | Contenido |
|---|---|
| `epen_variable_filtered.csv` | Dataset filtrado listo para *feature engineering* |
| `variables_kept.csv` | Resumen de variables conservadas |
| `variables_removed.csv` | Reporte de variables eliminadas por categoría |
| `high_missing_variables.csv` | Variables con alto porcentaje de nulos |
| `constant_variables.csv` | Variables constantes o casi constantes |

In [19]:
OUTPUT_PATH = Path('../data/processed')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# ── 1. Dataset filtrado ──────────────────────────────────────────────────────
out_filtered = OUTPUT_PATH / 'epen_variable_filtered.csv'
df_model_base.to_csv(out_filtered, index=False)
print(f"Guardado: {out_filtered}  ({df_model_base.shape[0]:,} × {df_model_base.shape[1]})")

# ── 2. Variables conservadas ─────────────────────────────────────────────────
vars_kept = pd.DataFrame({
    'variable'  : df_model_base.columns.tolist(),
    'tipo_dato' : df_model_base.dtypes.values.astype(str),
    'nulos'     : df_model_base.isnull().sum().values,
    'pct_nulos' : (df_model_base.isnull().sum() / len(df_model_base)).round(4).values,
    'n_unicos'  : df_model_base.nunique().values,
})
out_kept = OUTPUT_PATH / 'variables_kept.csv'
vars_kept.to_csv(out_kept, index=False)
print(f"Guardado: {out_kept}  ({len(vars_kept)} variables)")

# ── 3. Variables eliminadas por categoría ───────────────────────────────────
removed_categories = {
    'leakage'            : leakage_present,
    'administrativas'    : id_present,
    'expansion'          : ([expansion_col] if expansion_factor is not None else []),
    'duplicadas_exactas' : (duplicated_exact if 'duplicated_exact' in dir() else []),
    'alto_missing'       : (high_missing_safe_to_drop if 'high_missing_safe_to_drop' in dir() else []),
    'constantes'         : constant_variables,
    'alta_cardinalidad'  : hc_present,
}

removed_rows = [
    {'categoria': cat, 'variable': col}
    for cat, cols in removed_categories.items()
    for col in cols
]
df_removed = pd.DataFrame(removed_rows) if removed_rows else \
             pd.DataFrame(columns=['categoria', 'variable'])
out_removed = OUTPUT_PATH / 'variables_removed.csv'
df_removed.to_csv(out_removed, index=False)
print(f"Guardado: {out_removed}  ({len(df_removed)} variables eliminadas)")

# ── 4. Variables con alto missing ────────────────────────────────────────────
if 'missing_df' in dir() and 'high_missing_variables' in dir():
    high_miss_df = missing_df[missing_df['columna'].isin(high_missing_variables)]
else:
    high_miss_df = pd.DataFrame(columns=['columna', 'cantidad_nulos', 'porcentaje_nulos', 'tipo_dato'])

out_high_miss = OUTPUT_PATH / 'high_missing_variables.csv'
high_miss_df.to_csv(out_high_miss, index=False)
print(f"Guardado: {out_high_miss}  ({len(high_miss_df)} variables)")

# ── 5. Variables constantes / casi constantes ────────────────────────────────
const_rows = [{'columna': col, 'tipo': 'constante', 'n_unicos': 1, 'frecuencia_max': None}
              for col in constant_variables]
const_rows += [{'columna': item['columna'], 'tipo': 'casi_constante',
                'n_unicos': None, 'frecuencia_max': item['frecuencia_relativa']}
               for item in quasi_constant_variables]

const_df = pd.DataFrame(const_rows) if const_rows else \
           pd.DataFrame(columns=['columna', 'tipo', 'n_unicos', 'frecuencia_max'])
out_const = OUTPUT_PATH / 'constant_variables.csv'
const_df.to_csv(out_const, index=False)
print(f"Guardado: {out_const}  ({len(const_df)} variables)")

print("\nTodos los archivos guardados correctamente.")

Guardado: ..\data\processed\epen_variable_filtered.csv  (24,054 × 46)
Guardado: ..\data\processed\variables_kept.csv  (46 variables)
Guardado: ..\data\processed\variables_removed.csv  (30 variables eliminadas)
Guardado: ..\data\processed\high_missing_variables.csv  (12 variables)
Guardado: ..\data\processed\constant_variables.csv  (36 variables)

Todos los archivos guardados correctamente.


---
## 18. Conclusiones

### Resumen de acciones realizadas en este notebook:

| Acción | Detalle |
|---|---|
| Variables de *data leakage* eliminadas | `P209H`, `C333`, `C334` |
| Identificadores administrativos eliminados | `ANIO`, `MES`, `CONGLOMERADO`, `MUESTRA`, `SELVIV`, `HOGAR`, `LLAVE_PANEL`, `C201`, `C300n`, `NROINF` |
| Factor de expansión excluido como predictor | `fa_son24` (guardado como auxiliar) |
| Duplicados exactos eliminados | `INGTRABW` si era idéntica a `ingtrabw` |
| Variables de alta cardinalidad excluidas del modelo base | `C308_COD`, `C309_COD` |
| Variables constantes eliminadas | Columnas con un solo valor único |
| Variables con alto missing eliminadas | Columnas > 60% de nulos sin relevancia conceptual |

### Variables revisadas pero NO eliminadas automáticamente:
- Variables con alto porcentaje de nulos pero conceptualmente importantes (ingresos, horas, empleo): se mantienen para evaluación en *feature engineering* o *feature selection*.
- Variables casi constantes: documentadas para revisión en etapa siguiente.
- Pares con alta correlación: documentados para *feature selection* estadística.

### Próximos pasos:
1. **Feature Engineering** (`04_feature_engineering/`): creación de nuevas características a partir de las variables conservadas.
2. **Train/Test Split** (`04_feature_engineering/train_test_split.ipynb`): separación del dataset en entrenamiento y prueba.
3. **Feature Selection** (`05_feature_selection/`): selección estadística de las mejores variables para el modelo.

> El archivo `epen_variable_filtered.csv` es el punto de partida para las etapas siguientes.